# Day 01 · RAG QA 봇 실습 (로컬)
임베딩·벡터DB는 **로컬(CPU)**, 생성은 **NVIDIA build**(Qwen 모델, 개인 토큰). 7단계 RAG를 직접 만듭니다.

`적재 → 청킹 → 임베딩 → 저장 → 검색 → 증강 → 생성`

## 0 · 설치


In [ ]:
!pip install -q sentence-transformers qdrant-client langchain-text-splitters openai


## 1 · NVIDIA API 토큰 적용 🔑
생성(LLM)은 **NVIDIA build** 의 Qwen 모델을 씁니다. 개인 **NVIDIA API 토큰**(`nvapi-...`)이 필요해요.

- 토큰이 아직 없다면 → **`실습_시작하기.ipynb`** 에서 발급 (build.nvidia.com · 무료 · 카드 불필요)
- 아래 셀을 실행하면 토큰을 **직접 입력**받습니다(화면에 안 보임). 이미 `NVIDIA_API_KEY` 환경변수가 있으면 자동 사용.

> 임베딩·벡터DB는 로컬(CPU)이라 토큰이 필요 없고, **생성만** 토큰을 씁니다.

In [ ]:
import os, getpass
from openai import OpenAI

# 생성 엔드포인트: NVIDIA build (기본). 강사 DGX로 바꾸려면 아래 줄 주석 해제.
LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")

# NVIDIA API 토큰: 환경변수(NVIDIA_API_KEY) 있으면 자동, 없으면 직접 입력(화면 비표시)
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or os.getenv("QWEN_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")

# 사용 가능한 모델에서 Qwen 채팅 모델 자동 확정
_c = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)
_av = [m.id for m in _c.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl", "embed", "rerank"))]
    LLM_MODEL = _q[0] if _q else _av[0]
print("엔드포인트:", LLM_BASE_URL, "| 모델:", LLM_MODEL)

EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"   # 로컬 다국어 임베딩
QDRANT_PATH = "./qdrant_db"

## 2 · 문서 적재 (예시 코퍼스)
실제로는 사내 문서를 넣습니다. 여기선 작은 예시로 흐름을 익힙니다.


In [ ]:
docs = [
    "연차 휴가는 사용 3영업일 전까지 사내포털에서 신청하고 팀장 승인을 받아야 한다.",
    "비용 정산은 영수증을 첨부해 신청서를 작성하고, 경영지원팀 검토 후 재무팀이 최종 승인한다.",
    "재택근무는 주 2회까지 가능하며, 전날 18시까지 팀 채널에 사전 공유한다.",
    "보안 정책상 외부 USB 사용은 금지되며, 파일 공유는 사내 드라이브를 사용한다.",
    "에러코드 ERR-4021은 인증 토큰 만료를 의미하며, 재로그인으로 해결된다.",
]


## 3 · 청킹
문장이 짧아 그대로 두지만, 긴 문서는 RecursiveCharacterTextSplitter로 분할합니다.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks = []
for d in docs:
    chunks.extend(splitter.split_text(d))
print(len(chunks), "청크")
chunks[:3]


## 3.5 · [실습] 진짜 문서로 청킹 결과 눈으로 보기
위 예시는 문장이 짧아 그대로 남았지만, 실제로는 훨씬 긴 문서가 여러 청크로 잘립니다. 아래에서 조금 더 긴 문서(회사 정책 여러 개를 묶은 글)를 실제로 잘라보고, **청크 경계와 overlap이 어디에 생기는지** 눈으로 확인합니다.

In [ ]:
def visualize_chunks(text, chunk_size=400, chunk_overlap=60):
    sp = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    parts = sp.split_text(text)
    OVERLAP, RESET, HEAD = "\033[43m\033[30m", "\033[0m", "\033[1m"
    print(f"원문 {len(text)}자 → {len(parts)}개 청크  (chunk_size={chunk_size}, overlap={chunk_overlap})")
    print("=" * 60)
    for i, c in enumerate(parts):
        head_len = min(chunk_overlap, len(c)) if i > 0 else 0
        tail_len = min(chunk_overlap, len(c) - head_len) if i < len(parts) - 1 else 0
        head, tail = c[:head_len], (c[len(c) - tail_len:] if tail_len else "")
        mid = c[head_len: len(c) - tail_len] if tail_len else c[head_len:]
        print(f"\n{HEAD}[청크 {i+1}/{len(parts)} · {len(c)}자]{RESET}")
        print((OVERLAP + head + RESET if head else "") + mid + (OVERLAP + tail + RESET if tail else ""))
    print(f"\n{HEAD}(노란 배경 = 앞/뒤 청크와 겹치는 overlap 구간){RESET}")

long_doc = """연차 및 휴가 정책
연차 휴가는 입사 1년 차부터 매년 15일이 부여되며, 근속 연수에 따라 최대 25일까지 늘어난다. 연차는 사용 3영업일 전까지 사내포털의 '휴가 신청' 메뉴에서 신청하고 팀장의 승인을 받아야 한다. 반차 및 반반차 단위 사용도 가능하며, 이 경우에도 동일한 절차를 따른다. 승인되지 않은 상태로 연차를 사용하면 결근으로 처리될 수 있으니 반드시 승인 완료 여부를 확인해야 한다. 연차는 회계연도 기준으로 이월되지 않으며, 남은 연차는 12월 말까지 소진하는 것을 권장한다.

재택근무 정책
재택근무는 주 2회까지 가능하며, 전날 18시까지 소속 팀 채널에 사전 공유해야 한다. 재택근무 중에도 사내 보안 정책은 동일하게 적용되며, 사내 시스템에 접속할 때는 반드시 VPN을 사용해야 한다. 재택근무 신청이 반복적으로 지연되거나 사전 공유 없이 이루어질 경우, 팀장 재량으로 재택근무가 제한될 수 있다. 재택근무일에도 코어타임(오전 10시~오후 4시)에는 메신저 응답이 가능한 상태를 유지해야 한다.

비용 정산 절차
비용 정산은 영수증을 첨부하여 신청서를 작성하고, 경영지원팀의 1차 검토 후 재무팀이 최종 승인한다. 10만원 이상의 비용은 사전 승인이 필요하며, 사전 승인 없이 집행된 비용은 정산이 거부될 수 있다. 정산 신청은 지출일로부터 30일 이내에 완료해야 하며, 기한을 넘긴 건은 별도의 예외 승인 절차를 거쳐야 한다. 법인카드 사용 내역은 매월 5일까지 경영지원팀에 제출해야 한다.

보안 및 자료 공유 정책
보안 정책상 외부 USB 등 이동식 저장장치의 사용은 원칙적으로 금지되며, 모든 파일 공유는 사내 승인된 드라이브를 통해서만 이루어져야 한다. 외부 협력사와 자료를 공유할 때는 반드시 보안팀의 사전 승인을 받아야 하며, 승인되지 않은 채널(개인 이메일, 메신저 등)을 통한 자료 전송은 사규 위반에 해당한다. 사내 문서에는 보안 등급(대외비, 사내한정, 공개)을 반드시 표기해야 한다.

시스템 오류 코드 안내
에러코드 ERR-4021은 인증 토큰 만료를 의미하며, 재로그인을 통해 해결할 수 있다. ERR-5010은 네트워크 연결 문제를 의미하며, VPN 연결 상태를 먼저 확인해야 한다. ERR-6003은 권한 부족을 의미하며, 시스템 관리자에게 권한 신청을 해야 한다. 반복적으로 동일한 오류가 발생할 경우, IT지원팀에 문의하여 계정 상태를 점검받아야 한다.

신입사원 온보딩 절차
입사 첫날에는 인사팀에서 사내 계정과 장비를 지급하며, 보안 서약서 작성이 필수다. 입사 후 1주일 이내에 필수 온라인 교육(정보보안, 성희롱 예방)을 이수해야 한다. 수습 기간은 3개월이며, 수습 종료 전 팀장과 1차 면담을 진행한다."""

### 지금 쓰는 설정 그대로 (chunk_size=400, overlap=60)

In [ ]:
visualize_chunks(long_doc, chunk_size=400, chunk_overlap=60)

### chunk_size를 줄이면? (200자로 실습)
슬라이드의 '트레이드오프'를 직접 확인합니다 — 청크가 더 잘게, 더 많이 나뉘고 문맥이 잘릴 위험이 커집니다.

In [ ]:
visualize_chunks(long_doc, chunk_size=200, chunk_overlap=40)

## 4 · 임베딩 (로컬·CPU)
정규화하면 코사인=내적이라 검색이 간단해집니다.


In [4]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL)

def embed(texts):
    return embedder.encode(texts, normalize_embeddings=True)

vecs = embed(chunks)
print("임베딩 shape:", vecs.shape)   # (청크 수, 차원)


NameError: name 'EMBED_MODEL' is not defined

## 4.5 · [실습] 코사인 유사도 직접 확인하기 (라이브 데모)
슬라이드의 '유사도 예측' 미니실습을 실제 임베딩으로 검증합니다. `embed()`는 이미 정규화(`normalize_embeddings=True`)돼 있어 **내적 = 코사인 유사도**입니다.
아래 예제를 하나씩 실행하며 라이브로 설명해보세요 — 예측과 실제 숫자를 비교하는 게 핵심입니다.

In [2]:
import numpy as np

def show_similarity(anchor, candidates):
    vecs = embed([anchor] + candidates)
    anchor_vec, cand_vecs = vecs[0], vecs[1:]
    sims = cand_vecs @ anchor_vec   # 정규화된 벡터 → 내적 = 코사인 유사도
    order = np.argsort(-sims)
    print(f"[기준] {anchor}\n")
    for i in order:
        bar = "█" * round(sims[i] * 24)
        print(f"  {sims[i]:.3f} {bar}")
        print(f"        {candidates[i]}")
    print()

### 예제 1 · 슬라이드 미니실습 그대로 (환불)
예측: A·C가 가깝고 B는 멀 것 — 실제로 확인해봅니다.

In [3]:
show_similarity(
    "환불은 어떻게 받나요?",
    [
        "결제 취소 및 환불 절차 안내",
        "배송은 보통 2~3일 걸립니다",
        "반품 시 환불은 영업일 5일 내",
    ],
)

NameError: name 'embed' is not defined

### 예제 1-1 · 코사인 유사도 vs 내적 vs 유클리드 거리(L2)
슬라이드의 세 가지 유사도 척도(코사인·내적·L2)를 같은 예제로 직접 비교합니다.

In [ ]:
def compare_metrics(anchor, candidates):
    vecs = embed([anchor] + candidates)   # embed()는 이미 정규화됨 (normalize_embeddings=True)
    a, c = vecs[0], vecs[1:]
    cosine = c @ a                          # 정규화된 벡터 → 내적 = 코사인
    dot    = c @ a                          # (정규화 상태에서는 코사인과 완전히 같은 값)
    l2     = np.linalg.norm(c - a, axis=1)  # 유클리드 거리 — 작을수록 유사

    order = np.argsort(-cosine)
    print(f"[기준] {anchor}\n")
    print(f"{'순위':<4}{'후보':<32}{'cosine':>8}{'dot':>8}{'L2':>8}")
    for rank, i in enumerate(order, 1):
        print(f"{rank:<4}{candidates[i][:28]:<32}{cosine[i]:8.3f}{dot[i]:8.3f}{l2[i]:8.3f}")
    print(
        "\n정규화된 벡터에서는 cosine == dot product(완전히 같은 값)이고, "
        "L2² = 2 - 2·cosine 관계라서 L2가 작을수록 cosine이 큽니다 → 세 지표의 순위가 항상 일치합니다."
    )

compare_metrics(
    "환불은 어떻게 받나요?",
    [
        "결제 취소 및 환불 절차 안내",
        "배송은 보통 2~3일 걸립니다",
        "반품 시 환불은 영업일 5일 내",
    ],
)

### 예제 2 · 패러프레이즈 vs 관련 있지만 다른 의도 vs 완전 무관
같은 뜻을 다른 표현으로 쓴 문장(패러프레이즈)이 가장 가깝고, 주제는 겹치지만 의도가 다른 문장은 중간, 완전 무관한 문장은 가장 멉니다.

In [ ]:
show_similarity(
    "비밀번호를 잊어버렸어요",
    [
        "로그인 정보를 분실했습니다",   # 패러프레이즈 — 가장 가까울 것
        "회원 탈퇴는 어떻게 하나요",     # 계정 관련이지만 다른 의도
        "오늘 점심 메뉴는 뭐예요",       # 완전 무관
    ],
)

### 예제 3 · 표면(단어)은 겹쳐도 의미가 다른 함정
'배'라는 같은 단어가 들어가도 문맥에 따라 뜻이 다릅니다(복통 vs 선박). 실행해보면 진짜 의미가 같은 문장이 근소하게 1위지만, 단어만 겹치는 문장도 꽤 가깝게 따라붙는 걸 볼 수 있습니다 — **키워드만 믿으면 안 되는 이유**이자, Day 02의 Reranker가 필요한 이유입니다.

In [ ]:
show_similarity(
    "배가 아파요",
    [
        "복통 때문에 병원에 갔다",   # 의미는 같음, 단어는 안 겹침
        "배를 타고 섬에 다녀왔다",   # 단어 '배'는 겹치지만 의미(선박)는 다름
        "사과가 정말 맛있다",       # 완전 무관
    ],
)

### 예제 4 · 실제 코퍼스 문서와 연결
위에서 만든 Day 01 RAG 코퍼스(`docs`) 전체와 비교해서, 검색이 실제로 왜 되는지 유사도 숫자로 확인합니다.

In [ ]:
show_similarity(
    "에러코드 뜻이 뭔가요?",
    docs,   # 위에서 정의한 5개 사내 정책 문서 전체와 비교
)

### 예제 5 · 다국어(한국어 ↔ 영어)
오늘 쓰는 임베딩 모델(`paraphrase-multilingual-MiniLM-L12-v2`)은 다국어 지원 모델입니다. 짧고 애매한 문장은 '의미'보다 '언어'로 더 가깝게 묶이기도 하지만(직접 확인해보세요), 문장이 구체적일수록 언어가 달라도 의미로 매칭됩니다.

In [ ]:
show_similarity(
    "에러코드 ERR-4021은 인증 토큰 만료를 의미합니다",
    [
        "Error code ERR-4021 means the authentication token has expired",  # 다른 언어, 같은 의미
        "재택근무는 주 2회까지 가능합니다",                                    # 같은 언어, 무관한 주제
        "Remote work is allowed up to twice a week",                        # 다른 언어, 무관한 주제
    ],
)

## 5 · 벡터 DB 저장 (로컬 Qdrant)


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient(path=QDRANT_PATH)   # 로컬 파일 모드
dim = vecs.shape[1]
client.recreate_collection(
    collection_name="docs",
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
)
client.upsert("docs", points=[
    PointStruct(id=i, vector=v.tolist(), payload={"text": chunks[i]})
    for i, v in enumerate(vecs)
])
print("저장 완료:", client.count("docs").count, "청크")


## 6 · 검색 (Top-K)
질문을 임베딩해 가장 가까운 청크를 찾습니다. (`query_points` 사용)


In [ ]:
def search(query, k=3):
    qv = embed([query])[0]
    res = client.query_points("docs", query=qv.tolist(), limit=k).points
    return [(p.payload["text"], round(p.score, 3)) for p in res]

for t, s in search("연차 며칠 전에 신청해요?"):
    print(round(s,3) if isinstance(s,float) else s, "|", t)


## 7 · 증강 + 생성 (NVIDIA · Qwen)
검색된 근거만 사용하고, 없으면 '모른다'고 답하도록 프롬프트를 단단히 만듭니다.

In [ ]:
from openai import OpenAI
llm = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)   # NVIDIA build (Qwen)

SYSTEM = ("아래 [자료]의 내용만 근거로 답하라. 자료에 없으면 '문서에 없음'이라고만 답하라. "
          "답 끝에 (근거: 자료번호)를 표기하라. 추측 금지.")

def rag_answer(query, k=3):
    hits = search(query, k)
    context = "\n".join(f"[{i+1}] {t}" for i,(t,_) in enumerate(hits))
    msg = [{"role":"system","content":SYSTEM},
           {"role":"user","content":f"[자료]\n{context}\n\n[질문] {query}"}]
    out = llm.chat.completions.create(model=LLM_MODEL, messages=msg, temperature=0.2)
    return out.choices[0].message.content

print(rag_answer("연차 휴가는 며칠 전에 신청하나요?"))

## 8 · 테스트 & 환각 점검
문서에 **없는** 질문에 '문서에 없음'으로 답하는지 확인하세요.


In [ ]:
for q in ["비용 정산은 누가 최종 승인하나요?",
          "ERR-4021은 무슨 뜻인가요?",
          "사내 헬스장 운영시간은?"]:   # 마지막은 문서에 없음 → '문서에 없음' 기대
    print("Q:", q)
    print("A:", rag_answer(q))
    print("-"*50)


## 산출물 체크리스트
- ✅ 로컬 임베딩 + 로컬 Qdrant + **NVIDIA(Qwen) 생성**으로 RAG 동작
- ✅ 검색된 근거로만 답하고, 없으면 '문서에 없음'
- ✅ 내 문서로 `docs`만 바꾸면 나만의 QA 봇 완성